# Graph communities: cluster the graph and write a report for each cluster

This is **step 4 and step 5** of GraphRAG, built on the v2 graph from `03_knowlegde_graph.ipynb`:

| Step | Here |
|---|---|
| 4. Community detection & summarization | group related entities with **Louvain** (2 levels), then **qwen3-4b writes a report** for each group |
| 5. Vector indexing | upload the reports to your **Qdrant** (new collection `okf_communities`, same dense + BM25 + ColBERT setup as `okf_chunks`) |

`06_graph_query_hybrid.ipynb` uses them in step 6.

### Why communities?

Graph retrieval starts from entities in the question and walks to nearby facts. That works for "what score did GPT-4 get on the bar exam?". It doesn't work for **broad** questions like "which papers discuss hallucination?" or "what are the main risks of these models?". Those need an overview that no single fact or chunk holds.

Microsoft's GraphRAG solves this by writing those overviews **ahead of time**:
1. **Cluster** the graph: entities that share many facts end up in the same community ("RLHF, reward model, instruction tuning, SFT…").
2. Have the LLM **write a report** for each community: a title, a summary, and findings with citations.
3. At question time, search the **reports**, not just the facts.

The reports play the same role as the curated topic pages in your OKF bundle. The difference is that qwen3-4b writes them, from the extracted facts.

```
graph_facts_v2.jsonl ─► undirected graph ─► Louvain (level 0) ─► split big ones (level 1) ─► report units
                                                                                              │
                          Qdrant okf_communities ◄─ upload ◄─ graph_communities.jsonl ◄─ qwen3-4b writes a report per unit
```

## 1. Setup

- `RESOLUTION`: Louvain's knob. Higher means more, smaller communities.
- `MAX_COMMUNITY_SIZE`: communities bigger than this get split again (level 1).
- `MIN_COMMUNITY_SIZE`: smaller ones get no report. Their facts are still in the graph.
- `MAX_FACTS_PER_REPORT`: how many facts one LLM call sees. Hubs like GPT-4 have hundreds, so they're split by kind of fact (section 4).

In [10]:
import hashlib
import json
import os
import re
import time
import uuid
from collections import Counter, defaultdict
from pathlib import Path

import networkx as nx
import pandas as pd
import requests
from pydantic import BaseModel, Field

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FACTS_PATH = ARTIFACTS_DIR / "graph_facts_v2.jsonl"
ENTITIES_PATH = ARTIFACTS_DIR / "graph_entities_v2.csv"
REPORTS_PATH = ARTIFACTS_DIR / "graph_communities.jsonl"

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b-instruct-2507-q4_K_M"
NUM_CTX = 8192

INCLUDE_NEW_FACTS = True      # same switch as graph_query: use facts only Part 3's second check accepted
RESOLUTION = 1.0
SEED = 42                     # Louvain is random; a fixed seed gives the same communities every run
MAX_COMMUNITY_SIZE = 40
MIN_COMMUNITY_SIZE = 5
MAX_FACTS_PER_REPORT = 50

assert FACTS_PATH.exists(), "run 03_knowlegde_graph.ipynb Part 3 first"
print("project:", PROJECT_ROOT, "| networkx", nx.__version__)

project: c:\Users\Swapn\OneDrive\Documents\GitHub\OKF | networkx 3.6.1


In [11]:
# check: Ollama is running and the model is installed
tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json()
installed = [m["name"] for m in tags.get("models", [])]
assert MODEL_NAME in installed, f"{MODEL_NAME} not installed. Run: ollama pull {MODEL_NAME}"
print("ok:", MODEL_NAME)

ok: qwen3:4b-instruct-2507-q4_K_M


## 2. An undirected, weighted graph

Clustering asks one question: **which entities belong together?** Direction doesn't matter for that (`GPT-4 → OpenAI` links them just as much as `OpenAI → GPT-4`), so we use an undirected `nx.Graph`.

The **weight** of an edge is the total `support` of the facts between the two entities. A pair mentioned in 14 chunks is tied together much more strongly than a pair mentioned once.

In [12]:
facts = [json.loads(line) for line in FACTS_PATH.open(encoding="utf-8")]
if not INCLUDE_NEW_FACTS:
    facts = [f for f in facts if f["origin"] == "v1"]
entity_type = pd.read_csv(ENTITIES_PATH).set_index("name")["type"].to_dict()

U = nx.Graph()
for f in facts:
    w = U.get_edge_data(f["source"], f["target"], {"weight": 0})["weight"]
    U.add_edge(f["source"], f["target"], weight=w + f["support"])

strength = dict(U.degree(weight="weight"))          # weighted degree: how much each entity is talked about
print(f"facts: {len(facts)} | entities: {U.number_of_nodes()} | entity pairs: {U.number_of_edges()}")

facts: 2801 | entities: 2393 | entity pairs: 2710


## 3. Level 0: Louvain

Louvain starts with every entity in its own community, then keeps moving entities into the neighbouring community that raises **modularity** the most. Modularity measures "more edges inside communities than you'd expect by chance". It's built into NetworkX.

GraphRAG uses **Leiden**, an improved Louvain that guarantees every community is internally connected. NetworkX only exposes Leiden through extra backends, so we use Louvain. At this size the difference is small.

In [13]:
level0 = sorted(nx.community.louvain_communities(U, weight="weight", resolution=RESOLUTION, seed=SEED),
                key=len, reverse=True)
sizes = [len(c) for c in level0]
print(f"{len(level0)} communities | modularity {nx.community.modularity(U, level0, weight='weight'):.2f}")
print(f"sizes: {sizes[:12]} ... | {sum(s >= MIN_COMMUNITY_SIZE for s in sizes)} have >= {MIN_COMMUNITY_SIZE} entities "
      f"| {sum(s < MIN_COMMUNITY_SIZE for s in sizes)} are tiny (mostly 2-3 entity islands)")

296 communities | modularity 0.77
sizes: [375, 215, 178, 121, 109, 105, 97, 96, 89, 51, 44, 31] ... | 47 have >= 5 entities | 249 are tiny (mostly 2-3 entity islands)


In [14]:
# check: the biggest communities and their most-talked-about entities
def top_entities(nodes, n=8):
    return sorted(nodes, key=lambda x: (-strength[x], x))[:n]      # ties broken by name, so the order never changes


for i, c in enumerate(level0[:10]):
    print(f"C{i:<3} {len(c):4} entities: {', '.join(top_entities(c))}")

C0    375 entities: GPT-4, ChatGPT, hallucination, GPT-3.5, GSM8K, reasoning, law, programming
C1    215 entities: large language model, chain-of-thought prompting, in-context learning, HumanEval, mathematical reasoning, multimodal large language model, pre-training, MT-bench
C2    178 entities: foundation model, BERT, adaptation, natural language processing, search engine, ELMo, complexity, expressivity
C3    121 entities: The Pile, T5, BLOOM, PaLM, causal decoder, Gopher, disinformation, GPT-J
C4    109 entities: LLaMA, LLaMA 2, Self-Instruct-52K, Alpaca, FLAN-T5, parameter-efficient fine-tuning, Falcon, Vicuna
C5    105 entities: reinforcement learning from human feedback, Transformer, instruction tuning, supervised fine-tuning, reward model, GPT-1, human alignment, quantization
C6     97 entities: OpenAI, text-davinci-002, GLM, T0++, toxicity, Anthropic-LM v4-s3, TNLG v2, OPT
C7     96 entities: accuracy, robustness, Holistic Evaluation of Language Models, fairness, calibration, qu

## 4. Level 1: split the big communities

Communities bigger than `MAX_COMMUNITY_SIZE` get Louvain again, on just their own subgraph. That gives a **2-level hierarchy**: `C1` = "LLM techniques", and inside it `C1.0`, `C1.1`, … with narrower topics. GraphRAG does the same with more levels.

**Hubs need one more trick.** GPT-4 is linked to ~380 entities, and most of them link to nothing else. That's a star, and no clustering can split a star. So if a community still has more than `MAX_FACTS_PER_REPORT` facts, its facts are split **by kind**:

| facet | relations |
|---|---|
| `capabilities` | HAS_CAPABILITY, OUTPERFORMS, EVALUATED_ON, EVALUATES |
| `risks` | HAS_RISK, HAS_LIMITATION |
| `building` | USES, BUILDS_ON, TRAINED_ON, DEVELOPED_BY |
| `relations` | IS_A, PART_OF, COMPARED_WITH, MEASURES, COVERS |

So GPT-4 gets separate reports for its capabilities, its risks, and so on.

In [15]:
FACETS = {
    "capabilities": {"HAS_CAPABILITY", "OUTPERFORMS", "EVALUATED_ON", "EVALUATES"},
    "risks": {"HAS_RISK", "HAS_LIMITATION"},
    "building": {"USES", "BUILDS_ON", "TRAINED_ON", "DEVELOPED_BY"},
    "relations": {"IS_A", "PART_OF", "COMPARED_WITH", "MEASURES", "COVERS"},
}


def facts_inside(nodes):
    nodes = set(nodes)
    return [i for i, f in enumerate(facts) if f["source"] in nodes and f["target"] in nodes]


def best_facts(fact_ids, n=MAX_FACTS_PER_REPORT):
    # most supported first; ties -> facts about more central entities
    key = lambda i: (facts[i]["support"], strength[facts[i]["source"]] + strength[facts[i]["target"]])
    return sorted(fact_ids, key=key, reverse=True)[:n]


def leaf_units(cid, nodes, parent, level):
    # one report unit, or one per facet if it has too many facts for one LLM call
    fact_ids = facts_inside(nodes)
    if len(fact_ids) <= MAX_FACTS_PER_REPORT:
        return [{"id": cid, "level": level, "parent": parent, "kind": "leaf",
                 "members": top_entities(nodes, len(nodes)), "fact_ids": fact_ids}]
    units = []
    for facet, relations in FACETS.items():
        ids = [i for i in fact_ids if facts[i]["relation"] in relations]
        if len(ids) >= 3:
            ends = {facts[i]["source"] for i in ids} | {facts[i]["target"] for i in ids}
            units.append({"id": f"{cid}:{facet}", "level": level, "parent": parent, "kind": "facet",
                          "members": top_entities(ends, len(ends)), "fact_ids": best_facts(ids)})
    return units


def ordered_subgraph(nodes):
    # U.subgraph(set) would visit nodes in Python's set order, which changes between runs,
    # and then Louvain gives different sub-communities each time. Copying in U's own order keeps it stable.
    nodes = set(nodes)
    sub = nx.Graph()
    sub.add_nodes_from(n for n in U if n in nodes)
    sub.add_edges_from((u, v, d) for u, v, d in U.edges(data=True) if u in nodes and v in nodes)
    return sub


units = []                     # everything that gets a report
for i, community in enumerate(level0):
    cid = f"C{i}"
    if len(community) < MIN_COMMUNITY_SIZE:
        continue
    if len(community) <= MAX_COMMUNITY_SIZE:
        units += leaf_units(cid, community, parent=None, level=0)
        continue
    subs = sorted(nx.community.louvain_communities(ordered_subgraph(community), weight="weight",
                                                   resolution=RESOLUTION, seed=SEED), key=len, reverse=True)
    children = []
    for j, sub in enumerate(subs):
        if len(sub) >= MIN_COMMUNITY_SIZE:
            children += leaf_units(f"{cid}.{j}", sub, parent=cid, level=1)
    if children:
        units += children
        units.append({"id": cid, "level": 0, "parent": None, "kind": "parent",
                      "members": top_entities(community, len(community)),
                      "fact_ids": facts_inside(community), "children": [u["id"] for u in children]})

for u in units:
    u["fact_ids"] = sorted(u["fact_ids"])
    u["chunk_ids"] = sorted({c for i in u["fact_ids"] for c in facts[i]["chunk_ids"]})
    u["documents"] = dict(Counter(d for i in u["fact_ids"] for d in facts[i]["documents"]).most_common())
    # fingerprint: if the members or facts change, the report is re-written
    u["signature"] = hashlib.sha1(json.dumps([sorted(u["members"]), u["fact_ids"]]).encode()).hexdigest()[:12]
for u in units:                # a parent also changes when any of its children changes
    if u["kind"] == "parent":
        child_sigs = [x["signature"] for x in units if x["id"] in u["children"]]
        u["signature"] = hashlib.sha1((u["signature"] + "".join(child_sigs)).encode()).hexdigest()[:12]

kinds = Counter(u["kind"] for u in units)
print(f"report units: {len(units)}  ({dict(kinds)})")
print(f"entities covered by a leaf/facet report: {len({m for u in units if u['kind'] != 'parent' for m in u['members']})} of {U.number_of_nodes()}")

report units: 130  ({'facet': 11, 'leaf': 108, 'parent': 11})
entities covered by a leaf/facet report: 1625 of 2393


In [16]:
# check: the tree for the two biggest communities
for u in [u for u in units if u["kind"] == "parent"][:2]:
    print(f"{u['id']} ({len(u['members'])} entities): {', '.join(u['members'][:6])}")
    for child in u["children"]:
        c = next(x for x in units if x["id"] == child)
        print(f"   {c['id']:22} {c['kind']:6} {len(c['members']):4} entities, {len(c['fact_ids']):3} facts: {', '.join(c['members'][:5])}")

C0 (375 entities): GPT-4, ChatGPT, hallucination, GPT-3.5, GSM8K, reasoning
   C0.0:capabilities      facet   121 entities,  50 facts: GPT-4, GPT-3.5, programming, text-davinci-003, mathematics
   C0.0:risks             facet    95 entities,  50 facts: GPT-4, GPT-3.5, harmful content, overreliance, biased and unreliable content
   C0.0:building          facet    52 entities,  50 facts: GPT-4, smaller model, few-shot prompting, rule-based reward models, SUMMARIZE
   C0.0:relations         facet    14 entities,  12 facts: GPT-4, GPT-3.5, programming, text-davinci-003, mathematics
   C0.1                   leaf     45 entities,  48 facts: ChatGPT, GSM8K, Claude, WikiFact, deep learning
   C0.2                   leaf      7 entities,   6 facts: reasoning, non-ampliative reasoning, pattern matching, AutoGPT, long-form CoT data
   C0.3                   leaf      5 entities,   4 facts: hallucination, factual hallucination, knowledge faithfulness score, uncertainty estimation, uncertainty-bas

## 5. The report prompt

A **leaf** (or facet) report is written from its facts: one line per fact, with a short quote and the chunk IDs it came from.

A **parent** report is written from its children's reports, not from raw facts. That's how GraphRAG summarises big communities: summaries of summaries.

The output is forced into JSON with a pydantic schema, like the extraction in `03_knowlegde_graph.ipynb`. Each finding must cite chunk IDs, so every report sentence can be traced back to the paper text.

In [17]:
class Finding(BaseModel):
    text: str = Field(description="One sentence, a specific claim from the facts")
    chunk_ids: list[str] = Field(description="1-3 chunk ids (like GPT-4_0036) that support it")


class CommunityReport(BaseModel):
    title: str = Field(description="Short title, at most 8 words, naming what this group is about")
    summary: str = Field(description="2-4 sentences: what connects these entities and which papers discuss them")
    findings: list[Finding] = Field(description="3 to 6 key findings")


REPORT_SYSTEM = '''You write a short report about one community of a knowledge graph built from five AI research papers
(Sparks of AGI, GPT-4 Technical Report, HELM, On the Opportunities and Risks of Foundation Models, A Survey of LLMs).
Rules:
- Use ONLY the information given. No outside knowledge.
- Write 3 to 6 findings. Each finding is one specific sentence and cites 1-3 chunk ids copied from the input.
- The summary (2-4 sentences) says what connects the entities and which papers they come from.
- The title names the main topic of the group, at most 8 words.
- Keep numbers and model names exactly as written.
- The facts were extracted automatically; skip facts that look vague or wrong.

Answer with one JSON object and nothing else, with the keys in exactly this order:
{"findings": [{"chunk_ids": ["GPT-4_0036"], "text": "One specific sentence based on the facts."},
              {"chunk_ids": ["LLMs_0278", "LLMs_0279"], "text": "Another specific sentence."},
              {"chunk_ids": ["Holistic_Eval_0128"], "text": "A third specific sentence."}],
 "summary": "Two to four sentences on what connects these entities and which papers they come from.",
 "title": "Short topic title"}'''


PAPER_NAME = {"AGI": "Sparks of AGI", "GPT-4": "GPT-4 Technical Report", "Holistic_Eval": "HELM",
              "Foundation_Models": "Foundation Models report", "LLMs": "Survey of LLMs"}


def leaf_prompt(unit):
    lines = []
    for i in best_facts(unit["fact_ids"]):
        f = facts[i]
        quote = f["evidence"][0][:140] if f["evidence"] else ""
        lines.append(f"- {f['source']} --{f['relation']}--> {f['target']} | \"{quote}\" | {', '.join(f['chunk_ids'][:2])}")
    members = ", ".join(f"{m} ({entity_type.get(m, '?')})" for m in unit["members"][:25])
    papers = ", ".join(f"{PAPER_NAME[d]} ({n})" for d, n in unit["documents"].items())
    return f"Entities (most connected first): {members}\nPapers (facts per paper): {papers}\n\nFacts:\n" + "\n".join(lines)


def parent_prompt(unit, reports):
    parts = []
    for child in unit["children"]:
        r = reports.get(child)
        if not r or r.get("error"):
            continue
        findings = "\n".join(f"  - {x['text']} [{', '.join(x['chunk_ids'])}]" for x in r["findings"])
        parts.append(f"[{child}] {r['title']}: {r['summary']}\n{findings}")
    papers = ", ".join(f"{PAPER_NAME[d]} ({n})" for d, n in unit["documents"].items())
    return (f"This community contains {len(unit['members'])} entities, most connected first: "
            f"{', '.join(unit['members'][:15])}\nPapers: {papers}\n\nReports of its sub-communities:\n\n" + "\n\n".join(parts))


# The JSON shape Ollama must follow: the pydantic schema plus hard limits. "3 to 6" in a description
# is only a hint; minItems/maxItems are enforced while the model writes.
REPORT_SCHEMA = CommunityReport.model_json_schema()
REPORT_SCHEMA["properties"]["findings"].update(minItems=3, maxItems=6)
REPORT_SCHEMA["$defs"]["Finding"]["properties"]["chunk_ids"].update(minItems=1, maxItems=3)
MAX_REPORT_TOKENS = 1500      # a full report is ~500-900 tokens; more than this means the model is stuck


def write_report(prompt, timeout=300, attempts=3):
    problem = None
    for attempt in range(1, attempts + 1):
        # at temperature 0 a retry repeats the same reply, so retries get a little randomness
        temperature = 0 if attempt == 1 else 0.3 * (attempt - 1)
        response = requests.post(f"{OLLAMA_URL}/api/chat", timeout=timeout, json={
            "model": MODEL_NAME,
            "stream": False,
            "format": REPORT_SCHEMA,
            "options": {"temperature": temperature, "num_ctx": NUM_CTX, "num_predict": MAX_REPORT_TOKENS},
            "messages": [{"role": "system", "content": REPORT_SYSTEM},
                         {"role": "user", "content": prompt}],
        })
        response.raise_for_status()
        data = response.json()
        content = data["message"]["content"].strip()
        if data.get("done_reason") == "length":
            problem = (f"hit the {MAX_REPORT_TOKENS}-token cap with only blank output" if not content
                       else f"hit the {MAX_REPORT_TOKENS}-token cap mid-JSON (...{content[-60:]!r})")
        elif not content:
            problem = f"empty reply (done_reason={data.get('done_reason')})"
        else:
            try:
                report = CommunityReport.model_validate_json(content)
                report.findings = report.findings[:6]          # in case this Ollama version ignores maxItems
                return report
            except Exception as exc:
                problem = f"invalid JSON ({str(exc).splitlines()[0][:80]})"
        print(f"   attempt {attempt}/{attempts}: {problem}")
        time.sleep(2)
    raise ValueError(f"no usable report after {attempts} attempts: {problem}")

In [18]:
# check: the prompt for one community, then one real report
example = next(u for u in units if u["kind"] == "leaf" and 8 <= len(u["fact_ids"]) <= 20)
prompt = leaf_prompt(example)
print(example["id"], f"({len(prompt)} chars)\n")
print(prompt[:1500], "\n")

start = time.time()
report = write_report(prompt)
print(f"took {time.time() - start:.1f}s\n")
print(json.dumps(report.model_dump(), indent=2, ensure_ascii=False))

C1.2 (2834 chars)

Entities (most connected first): HumanEval (BENCHMARK), pre-training (CONCEPT), code synthesis (TASK), quality (METRIC), Muffin (DATASET), productivity (CONCEPT), retrieved documents (CONCEPT), conversation (TASK), mean log pass rate (METRIC), output per unit input (CONCEPT), pass rate (METRIC), pass@10 (METRIC), proficiency (DOMAIN), programming logic (DOMAIN), relevance (CONCEPT)
Papers (facts per paper): Survey of LLMs (8), Sparks of AGI (2), GPT-4 Technical Report (2), Foundation Models report (2)

Facts:
- HumanEval --COVERS--> code synthesis | "HumanEval [105]          Base/Fine-tuned/Specialized       Code synthesis           Human annotation" | LLMs_0474
- HumanEval --COVERS--> proficiency | "a docstring-to-code dataset consisting of 164 coding problems that test various aspects of programming logic and proficiency" | AGI_0076
- HumanEval --COVERS--> programming logic | "a docstring-to-code dataset consisting of 164 coding problems that test various aspects o

took 16.0s

{
  "title": "Code Synthesis and Retrieval in LLMs",
  "summary": "The community connects HumanEval, which evaluates code synthesis and programming logic, to Muffin, which includes code synthesis and conversation tasks, through shared tasks and datasets. Pre-training on code data enhances code synthesis, and quality of retrieved documents affects generated outputs. These connections are drawn from the Survey of LLMs, Sparks of AGI, and the Foundation Models report.",
  "findings": [
    {
      "text": "HumanEval covers code synthesis and proficiency through programming logic in coding problems.",
      "chunk_ids": [
        "LLMs_0474"
      ]
    },
    {
      "text": "Pre-training on code data improves code synthesis, and Muffin includes both code synthesis and conversation tasks.",
      "chunk_ids": [
        "LLMs_0418",
        "LLMs_0099"
      ]
    },
    {
      "text": "Relevance and quality of retrieved documents both impact generated answers.",
      "chunk_

## 6. Write every report (checkpointed)

Same pattern as the extraction run:
- each report is saved to `artifacts/graph_communities.jsonl` as soon as it's written
- you can stop the cell and re-run it any time; it continues where it stopped
- if you change the settings and a community's members change, its `signature` changes and only that report is re-written

Leaves and facets go first, then parents, because a parent needs its children's reports.

Cited chunk IDs are checked: an ID that isn't in that community's facts is dropped, so a report can't cite something it never saw.

In [19]:
def load_reports():
    if not REPORTS_PATH.exists():
        return {}
    with REPORTS_PATH.open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return {r["id"]: r for r in rows}                  # the latest row per id wins


def run_reports(units):
    reports = load_reports()
    done = lambda u: u["id"] in reports and reports[u["id"]]["signature"] == u["signature"] and not reports[u["id"]]["error"]
    todo = [u for u in units if u["kind"] != "parent" and not done(u)] + [u for u in units if u["kind"] == "parent" and not done(u)]
    print(f"{len(units) - len(todo)} already written, {len(todo)} to go")

    for n, u in enumerate(todo, start=1):
        start = time.time()
        row = {k: u[k] for k in ("id", "level", "parent", "kind", "members", "fact_ids", "chunk_ids", "documents", "signature")}
        row["children"] = u.get("children", [])
        try:
            prompt = parent_prompt(u, reports) if u["kind"] == "parent" else leaf_prompt(u)
            r = write_report(prompt)
            allowed = set(u["chunk_ids"])
            findings = [{"text": x.text, "chunk_ids": [c for c in x.chunk_ids if c in allowed]} for x in r.findings]
            row.update(title=r.title, summary=r.summary, findings=findings, error=None)
        except Exception as exc:
            row.update(title="", summary="", findings=[], error=str(exc)[:300])
        reports[u["id"]] = row
        with REPORTS_PATH.open("a", encoding="utf-8") as f:          # checkpoint
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"[{n}/{len(todo)}] {u['id']:24} {len(u['fact_ids']):3} facts  {time.time() - start:5.1f}s  "
              + (f"ERROR {row['error'][:60]}" if row["error"] else row["title"][:60]))
    return load_reports()


reports = run_reports(units)

0 already written, 130 to go
[1/130] C0.0:capabilities         50 facts   10.6s  GPT-4 Capabilities Across Domains
[2/130] C0.0:risks                50 facts   10.5s  GPT-4 Risks and Limitations
[3/130] C0.0:building             50 facts    9.0s  GPT-4 Operational Methods
[4/130] C0.0:relations            12 facts    8.2s  GPT-4 and Multimodal Capabilities
[5/130] C0.1                      48 facts    8.9s  ChatGPT Capabilities and Evaluations
[6/130] C0.2                       6 facts    9.3s  Reasoning in AI Systems
[7/130] C0.3                       4 facts    8.3s  Hallucination Detection in LLMs
[8/130] C0.4                       4 facts    7.1s  Supervised Learning and Continuous Prompts
[9/130] C1.0:capabilities         21 facts    7.8s  LLM Capabilities and Evaluations
[10/130] C1.0:risks                29 facts    8.2s  LLM Limitations in Specialized Contexts
[11/130] C1.0:building             30 facts    8.3s  LLM Methods and Numerical Tasks
[12/130] C1.0:relations           

In [20]:
# check: is every report written?
reports = load_reports()
current = {u["id"]: u for u in units}
ok = [r for i, r in reports.items() if i in current and r["signature"] == current[i]["signature"] and not r["error"]]
errors = [i for i, r in reports.items() if i in current and r["error"]]
print(f"reports: {len(ok)} of {len(units)} | errors: {errors or 'none'}")
print("re-run the cell above to retry errors" if len(ok) < len(units) else "all reports written")

reports: 130 of 130 | errors: none
all reports written


## 7. Read the reports

The reports are LLM-written, so read some. Good signs: a title that names a real topic, findings with numbers, citations on every finding. Bad signs: a vague title ("Various AI Concepts"), or findings with no `chunk_ids` (the model cited IDs it wasn't given, and they were dropped).

In [21]:
def show_report(rid):
    r = reports[rid]
    print(f"[{rid}] {r['title']}   ({r['kind']}, {len(r['members'])} entities, {len(r['fact_ids'])} facts)")
    print("   " + r["summary"])
    for x in r["findings"]:
        print(f"   - {x['text']}  {x['chunk_ids']}")
    print()


for r in sorted(ok, key=lambda r: -len(r["fact_ids"]))[:6]:
    show_report(r["id"])

[C0] GPT-4 Capabilities and Reasoning   (parent, 375 entities, 414 facts)
   This community connects through the capabilities, risks, and reasoning mechanisms of GPT-4 and related models, primarily drawn from the papers Sparks of AGI, GPT-4 Technical Report, Survey of LLMs, HELM, and Foundation Models report. Entities such as GPT-4, ChatGPT, reasoning, and hallucination are linked via performance evaluations, domain-specific tasks, and technical methods described in these papers. The connections span capabilities in programming and mathematics, risks in content generation, and foundational reasoning processes like pattern matching and non-ampliative reasoning.
   - GPT-4 is a large multimodal model capable of processing image and text inputs and producing text outputs.  ['GPT-4_0003']
   - ChatGPT is evaluated on GSM8K and Colored Objects for complex reasoning tasks and has the capability to perform conversational recommendation and general recommendation through specific prompts.  ['L

In [22]:
# check: quality signals for all reports
table = pd.DataFrame([{
    "id": r["id"], "kind": r["kind"], "title": r["title"][:50], "entities": len(r["members"]),
    "facts": len(r["fact_ids"]), "findings": len(r["findings"]),
    "uncited findings": sum(not x["chunk_ids"] for x in r["findings"]),
} for r in ok])
print(f"findings: {table['findings'].sum()} | without a valid citation: {table['uncited findings'].sum()}")
display(table.sort_values("facts", ascending=False).head(25))

findings: 479 | without a valid citation: 9


,id,kind,title,entities,facts,findings,uncited findings
119,C0,parent,GPT-4 Capabilities and Reasoning,375,414,3,0
120,C1,parent,LLM Reasoning and Evaluation,215,236,3,0
121,C2,parent,Foundation Models and Their Capabilities,178,187,3,0
125,C6,parent,Model Performance and Bias in LLMs,97,183,3,3
122,C3,parent,LLM Architectural and Training Patterns,121,160,3,3
123,C4,parent,LLaMA Model Lineage and Instruction Datasets,109,146,3,0
124,C5,parent,Training and Alignment Methods in LLMs,105,119,3,0
126,C7,parent,Evaluation Metrics in Foundation Models,96,117,6,0
127,C8,parent,Evolution and Evaluation of Language Models,89,100,3,1
0,C0.0:capabilities,facet,GPT-4 Capabilities Across Domains,121,50,3,0


## 8. Step 5: upload the reports to Qdrant

Same setup as `okf_chunks` in `01_Chunking.ipynb`:
- **dense** (`all-MiniLM-L6-v2`) for meaning, **sparse** (`qdrant/bm25`) for keywords, **multi** (ColBERT) for reranking
- Qdrant Cloud computes the embeddings (`cloud_inference=True`), so nothing runs on your GPU
- a `content_hash` makes re-runs cheap: only new or changed reports are uploaded, and reports that no longer exist are deleted

It's a **new, separate collection** (`okf_communities`). `okf_chunks` is not touched.

In [23]:
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

load_dotenv(PROJECT_ROOT / ".env")
QDRANT_URL, QDRANT_API_KEY = os.getenv("QDRANT_URL"), os.getenv("QDRANT_API_KEY")
assert QDRANT_URL and QDRANT_API_KEY, "QDRANT_URL / QDRANT_API_KEY missing in .env"

COMMUNITY_COLLECTION = "okf_communities"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "qdrant/bm25"
LATE_INTERACTION_MODEL = "answerdotai/answerai-colbert-small-v1"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, cloud_inference=True, timeout=120)
if not client.collection_exists(COMMUNITY_COLLECTION):
    client.create_collection(
        COMMUNITY_COLLECTION,
        vectors_config={
            "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
            "multi": models.VectorParams(size=96, distance=models.Distance.COSINE,
                                         multivector_config=models.MultiVectorConfig(comparator=models.MultiVectorComparator.MAX_SIM),
                                         hnsw_config=models.HnswConfigDiff(m=0)),
        },
        sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    print(f"created collection '{COMMUNITY_COLLECTION}'")
else:
    print(f"collection '{COMMUNITY_COLLECTION}' exists")

created collection 'okf_communities'


In [24]:
def report_text(r):
    # what gets embedded: title + summary + findings + main entities
    findings = "\n".join(f"- {x['text']}" for x in r["findings"])
    return f"{r['title']}\n\n{r['summary']}\n\n{findings}\n\nEntities: {', '.join(r['members'][:15])}"


def point_id(rid):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"community:{rid}"))


def content_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


stored, offset = {}, None                      # point id -> content_hash already in Qdrant
while True:
    points, offset = client.scroll(COMMUNITY_COLLECTION, limit=256, offset=offset,
                                   with_payload=["content_hash"], with_vectors=False)
    stored.update({str(p.id): (p.payload or {}).get("content_hash") for p in points})
    if offset is None:
        break

wanted = {point_id(r["id"]): r for r in ok}
to_upload = [r for pid, r in wanted.items() if stored.get(pid) != content_hash(report_text(r))]
stale = [pid for pid in stored if pid not in wanted]
print(f"reports: {len(wanted)} | to upload: {len(to_upload)} | unchanged: {len(wanted) - len(to_upload)} | stale: {len(stale)}")

if stale:
    client.delete(COMMUNITY_COLLECTION, points_selector=models.PointIdsList(points=stale))
for i in range(0, len(to_upload), 32):
    batch = to_upload[i:i + 32]
    client.upsert(COMMUNITY_COLLECTION, wait=True, points=[
        models.PointStruct(
            id=point_id(r["id"]),
            vector={
                "dense": models.Document(text=report_text(r), model=DENSE_MODEL),
                "sparse": models.Document(text=report_text(r), model=SPARSE_MODEL),
                "multi": models.Document(text=report_text(r), model=LATE_INTERACTION_MODEL),
            },
            payload={"community_id": r["id"], "kind": r["kind"], "level": r["level"], "parent": r["parent"],
                     "title": r["title"], "summary": r["summary"], "findings": r["findings"],
                     "members": r["members"][:30], "documents": r["documents"], "chunk_ids": r["chunk_ids"],
                     "content_hash": content_hash(report_text(r))},
        ) for r in batch])
print("points in Qdrant:", client.count(COMMUNITY_COLLECTION, exact=True).count)

reports: 130 | to upload: 130 | unchanged: 0 | stale: 0
points in Qdrant: 130


In [25]:
# check: search the reports the same way 01_Chunking.ipynb searches chunks
def search_reports(query, top_k=3):
    return client.query_points(
        COMMUNITY_COLLECTION,
        prefetch=[models.Prefetch(query=models.Document(text=query, model=DENSE_MODEL), using="dense", limit=20),
                  models.Prefetch(query=models.Document(text=query, model=SPARSE_MODEL), using="sparse", limit=20)],
        query=models.Document(text=query, model=LATE_INTERACTION_MODEL),
        using="multi", limit=top_k, with_payload=["community_id", "title"],
    ).points


for q in ["Which papers discuss hallucination?", "What are the main risks of GPT-4?", "How is RLHF used to align models?"]:
    print(q)
    for p in search_reports(q):
        print(f"   {p.score:6.2f}  [{p.payload['community_id']}] {p.payload['title']}")

Which papers discuss hallucination?
     9.43  [C0.3] Hallucination Detection in LLMs
     9.40  [C0] GPT-4 Capabilities and Reasoning
     9.34  [C27] Tool and Component Interactions in LLMs
What are the main risks of GPT-4?
    13.50  [C0.0:risks] GPT-4 Risks and Limitations
    13.27  [C36] Visual Reasoning Risks in LLMs
    13.27  [C0] GPT-4 Capabilities and Reasoning
How is RLHF used to align models?
    13.36  [C5.0] RLHF and Alignment in LLMs
    13.35  [C5] Training and Alignment Methods in LLMs
    13.02  [C5.4] Reward Models and Alignment Methods


### Next

`06_graph_query_hybrid.ipynb` puts it all together (step 6): Qdrant chunk search + graph walk + these community reports.

Things to know:
- Re-running this notebook with the same settings costs nothing: reports and uploads are skipped when unchanged.
- If you change `RESOLUTION` or `MAX_COMMUNITY_SIZE`, the community IDs change. The changed reports get re-written, and old ones are deleted from Qdrant on the next upload.